<a href="https://colab.research.google.com/github/dodi-ctrl/PhishingDetector/blob/main/Metadata_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install dependencies
!pip install -q datasets scikit-learn pandas numpy matplotlib seaborn joblib
print("Installation OK")

In [ ]:
# Clone the repo to get feature_extraction.py, dataset_handling.py, metadata_agent.py
import os

if not os.path.exists('PhishingDetector'):
    !git clone https://github.com/dodi-ctrl/PhishingDetector.git

os.chdir('PhishingDetector')
print("Working directory:", os.getcwd())
print("Files:", os.listdir('.'))

In [ ]:
# Load dataset
from dataset_handling import load_meajor_dataset, extract_features_from_emails
from feature_extraction import FeatureExtractor

TOTAL_SAMPLES = 20000
LEGIT_RATIO   = 0.7   # 70% legitimate, 30% phishing

df, label_col = load_meajor_dataset(
    total_samples=TOTAL_SAMPLES,
    legit_ratio=LEGIT_RATIO
)

In [ ]:
# Identify text column and extract features
text_column = None
for col in ['text', 'body', 'email_body', 'content', 'email', 'Email Text']:
    if col in df.columns:
        text_column = col
        break

print(f"Using text column: '{text_column}'")

extractor   = FeatureExtractor()
features_df = extract_features_from_emails(df, text_column, extractor)
labels      = df[label_col].values

In [ ]:
# Train / test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features_df, labels,
    test_size=0.3,
    random_state=42,
    stratify=labels
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")
print(f"  - Legitimate:   {(y_test == 0).sum()}")
print(f"  - Phishing:     {(y_test == 1).sum()}")

In [ ]:
# Train the Metadata Agent
from metadata_agent import MetadataAgent

agent = MetadataAgent(n_estimators=100, max_depth=20, random_state=42)
agent.train(X_train, y_train, validate=True, tune_hyperparameters=False)

In [ ]:
# Evaluate — prints accuracy, precision, recall, F1, ROC-AUC,
# FPR, FNR, TNR, confusion matrix, classification report, and plots
results = agent.evaluate(X_test, y_test, plot_results=True)

In [ ]:
# Save model and results
agent.save_model('metadata_agent.pkl')
agent.export_results(results, 'metadata_agent_results.json')

print("\n" + "="*70)
print("Training complete!")
print(f"  Accuracy:  {results['accuracy']:.4f}")
print(f"  F1-Score:  {results['f1_score']:.4f}")
print(f"  ROC-AUC:   {results['roc_auc']:.4f}")
print("="*70)